## BMERA Validation Notebook

Detailed checks moved out of the main notebook: output parity, gradient-vs-finite-difference sweeps, and debug tensor inspections.

### Setup

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import pennylane as qml
from pennylane import numpy as qnp

from bmera_core import (
    density_matrix,
    compute_all_expval,
    compute_expval_real,
    compute_grad,
    model_output,
    compute_model_grad,
    loss,
    loss_and_grad_cached,
    weights,
)

jax.config.update('jax_enable_x64', True)
np.random.seed(11)
from bmera import build_bmera_meta
from bmera import cache_profile, forward_single_with_cache


### Gradient vs Finite Difference Sweep (types 1/2/3)

In [ ]:
def finite_diff_grad(fn, x, eps=2e-6):
    x = np.array(x, dtype=float)
    
    g = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        xp = x.copy(); xm = x.copy()
        xp[idx] += eps; xm[idx] -= eps
        g[idx] = (fn(xp) - fn(xm)) / (2 * eps)
        it.iternext()
    return g

def rel_err(a, b):
    return np.linalg.norm((a - b).ravel()) / (np.linalg.norm(a.ravel()) + np.linalg.norm(b.ravel()) + 1e-12)

rows = []
for type_, pcount in [(1, 4), (2, 5), (3, 5)]:
    for n_qubits in [4, 8]:
        a = int(np.log2(n_qubits))
        params = np.random.uniform(-np.pi, np.pi, (a * n_qubits, pcount))
        in_data = np.random.normal(0, 1, n_qubits)
        w = np.random.uniform(0, 1, n_qubits)

        g = np.array(compute_grad(type_, params, in_data, n_qubits, 1))
        g_fd = finite_diff_grad(lambda p: float(compute_expval_real(type_, p, in_data, n_qubits, 1)), params)

        gmp, gmw = compute_model_grad(type_, params, w, in_data, n_qubits)
        gmp_fd = finite_diff_grad(lambda p: float(model_output(type_, p, w, in_data, n_qubits)), params)

        rows.append((type_, n_qubits, rel_err(g, g_fd), rel_err(np.array(gmp), gmp_fd), np.linalg.norm(np.array(gmw))))

rows

### Cached Gradient vs Autodiff (Model Loss)


In [ ]:
for type_ in [1, 2, 3]:
    for n_qubits in [4, 8]:
        a = int(np.log2(n_qubits))
        n_per = 4 if type_ == 1 else 5
        params = jnp.array(np.random.uniform(-np.pi, np.pi, (a * n_qubits, n_per)))
        X = jnp.array(np.random.normal(0, 1, (4, n_qubits)))
        y = jnp.array(np.random.uniform(-1, 1, 4))
        w = jnp.array(np.random.uniform(0, 1, n_qubits))
        meta = build_bmera_meta(n_qubits, type_)

        l_auto, (gpa, gwa) = jax.value_and_grad(loss, (0, 4))(params, X, y, meta, w)
        l_cached, gpc, gwc = loss_and_grad_cached(params, X, y, meta, w)

        rel_p = np.linalg.norm(np.array(gpa - gpc).ravel()) / (np.linalg.norm(np.array(gpa).ravel()) + 1e-12)
        rel_w = np.linalg.norm(np.array(gwa - gwc).ravel()) / (np.linalg.norm(np.array(gwa).ravel()) + 1e-12)
        print(f'type={type_} n={n_qubits} loss_diff={float(abs(l_auto-l_cached)):.3e} rel_p={rel_p:.3e} rel_w={rel_w:.3e}')


### Pennylane Parity Checks Across Sizes

In [ ]:
from bmera.backends.pennylane import pennylane_bmera_expvals


errs = []
for type_, pcount in [(1, 4), (2, 5), (3, 5)]:
    for n_qubits in [4, 8]:
        a = int(np.log2(n_qubits))
        params = np.random.uniform(-np.pi, np.pi, (a * n_qubits, pcount))
        in_data = np.random.normal(0, 1, n_qubits)
        tn_out = np.array(compute_all_expval(type_, params, in_data, n_qubits))
        pl_out = pennylane_bmera_expvals(in_data, n_qubits, type_, params)
        np.testing.assert_allclose(tn_out, pl_out, atol=1e-9, rtol=1e-9)
        errs.append((type_, n_qubits, float(np.max(np.abs(tn_out - pl_out)))))
errs

### Debug Tensor/Rho Inspection

In [ ]:
rho = density_matrix(np.random.uniform(0, 1, tuple(np.int32(np.ones(6) * 2))), np.array([0, 2, 3]))
rho_trace = rho.trace_rho(np.array([2]))
rho_trace.tensor.shape, rho_trace.qubits

In [ ]:
meta = build_bmera_meta(4, 1)
params = np.zeros((8, 4))
expvals, cache = forward_single_with_cache(params, np.zeros(4), meta)
cache_profile(cache, meta)['total_cache_nbytes']